# Selectivity Analysis

Implements Design Doc §5.6 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 5, using data isolated in Phase 2 and the anchor compounds from §4.3.

**Design Doc §5.6/§9 caveat, stated up front:** this whole notebook is a validation exercise on a handful of named anchor compounds, not a standalone trained selectivity classifier -- the paired data volume doesn't support one with any real confidence.

This notebook is being built incrementally, one plan step at a time. **This pass covers only step 1: identifying compounds with both WT and D816V records and computing the selectivity ratio.** Applying the Phase 4/addendum potency model to WT/mutant subsets (step 2) and the anchor-point check (step 3) come in later passes.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from data_utils import get_variant_pairs

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df["kit_variant"].value_counts()

Loaded: (5565, 10)


kit_variant
WT       3839
D816V    1726
Name: count, dtype: int64

## 1. Identify paired compounds and compute the selectivity ratio

Design Doc §5.6: for any compound with both wild-type and D816V bioactivity records, compute a selectivity ratio (wild-type potency / mutant potency).

`get_variant_pairs` (in [`src/data_utils.py`](../src/data_utils.py)) pivots the cleaned table to one row per compound with *both* variants measured, and computes:

- `log_selectivity` = `p_value_wt - p_value_d816v` (difference in log-molar units)
- `selectivity_ratio` = `10 ** log_selectivity` = WT potency / D816V potency

`selectivity_ratio > 1` means more potent against WT (loses efficacy against the mutant, the imatinib pattern); `< 1` means more potent against the mutant; `≈ 1` means comparable potency against both (the dasatinib pattern, per §4.3).

In [2]:
pairs = get_variant_pairs(df)
print(f"{len(pairs)} compounds have both a WT and a D816V record")
pairs.head()

925 compounds have both a WT and a D816V record


,canonical_smiles,p_value_wt,p_value_d816v,censored_wt,censored_d816v,log_selectivity,selectivity_ratio,both_censored
molecule_chembl_id,,,,,,,,
CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,5.000000,True,True,0.000000,1.000000,True
CHEMBL101253,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,5.000000,False,True,1.677781,47.619048,False
CHEMBL103667,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCCN3CCO...,6.585027,4.522879,False,True,2.062148,115.384615,False
CHEMBL105442,O=C(NOCC1CC1)c1ccc(F)c(F)c1Nc1ccc(I)cc1Cl,5.000000,5.000000,True,True,0.000000,1.000000,True
CHEMBL1171364,COc1ccc(/C=C2\Oc3cc(O)ccc3C2=O)cc1,4.806875,5.853872,False,False,-1.046997,0.089744,False


### Sanity checks

Confirm the table is well-formed before using it for anything -- no missing ratios, no duplicate compounds, and the count matches what earlier phases already found (Phase 4's notebook flagged 925 compounds with both variants when discussing the scaffold split).

In [3]:
assert pairs.index.is_unique, "Duplicate compound in the paired table"
assert pairs["selectivity_ratio"].notna().all(), "Missing selectivity ratios"
assert (pairs["selectivity_ratio"] > 0).all(), "Selectivity ratio should always be positive (it's a 10**x)"
assert len(pairs) == 925, f"Expected 925 paired compounds (per Phase 4's own count), got {len(pairs)}"

print(f"Verified: {len(pairs)} paired compounds, unique index, no missing/non-positive ratios.")
pairs["selectivity_ratio"].describe()

Verified: 925 paired compounds, unique index, no missing/non-positive ratios.


count     925.000000
mean       30.921104
std       122.954804
min         0.000930
25%         0.181818
50%         1.000000
75%        10.000000
max      1790.287185
Name: selectivity_ratio, dtype: float64

### Distribution overview

A quick look at how selectivity skews across the dataset, before zooming into the two named anchors.

In [4]:
more_wt_potent = (pairs["selectivity_ratio"] > 2).sum()
more_mutant_potent = (pairs["selectivity_ratio"] < 0.5).sum()
comparable = len(pairs) - more_wt_potent - more_mutant_potent

print(f"More potent against WT (ratio > 2, i.e. >2-fold):      {more_wt_potent} ({more_wt_potent/len(pairs):.1%})")
print(f"More potent against D816V (ratio < 0.5, i.e. >2-fold): {more_mutant_potent} ({more_mutant_potent/len(pairs):.1%})")
print(f"Comparable potency (0.5x-2x):                          {comparable} ({comparable/len(pairs):.1%})")

More potent against WT (ratio > 2, i.e. >2-fold):      348 (37.6%)
More potent against D816V (ratio < 0.5, i.e. >2-fold): 255 (27.6%)
Comparable potency (0.5x-2x):                          322 (34.8%)


### Data-quality caveat: censoring can fake a "comparable potency" ratio

A ratio near 1.0 is only real evidence of comparable potency if both p_values are actual measurements. If both sides are instead capped at the same censoring bound (Phase 2: e.g. both ">10000 nM" -> both capped to `p_value=5.0`), the ratio lands at exactly 1.0 as an artifact of the shared cap, not because the compound is truly equipotent against both variants. `both_censored` (from `get_variant_pairs`) flags these rows explicitly rather than letting them silently inflate the "comparable" bucket above.

In [5]:
n_both_censored = pairs["both_censored"].sum()
n_both_censored_ratio_one = ((pairs["both_censored"]) & (pairs["selectivity_ratio"] == 1.0)).sum()
n_either_censored = (pairs["censored_wt"] | pairs["censored_d816v"]).sum()

print(f"Both sides censored: {n_both_censored} / {len(pairs)} ({n_both_censored/len(pairs):.1%})")
print(f"  Of those, ratio == exactly 1.0 (same cap both sides, uninformative): {n_both_censored_ratio_one}")
print(f"At least one side censored: {n_either_censored} / {len(pairs)} ({n_either_censored/len(pairs):.1%})")

# Recompute the distribution excluding the uninformative both-censored-same-cap rows.
informative = pairs[~pairs["both_censored"]]
more_wt_potent = (informative["selectivity_ratio"] > 2).sum()
more_mutant_potent = (informative["selectivity_ratio"] < 0.5).sum()
comparable = len(informative) - more_wt_potent - more_mutant_potent
print(f"\nExcluding both-censored rows ({len(informative)} compounds remain):")
print(f"  More potent against WT (>2-fold):      {more_wt_potent} ({more_wt_potent/len(informative):.1%})")
print(f"  More potent against D816V (>2-fold):   {more_mutant_potent} ({more_mutant_potent/len(informative):.1%})")
print(f"  Comparable potency (0.5x-2x):           {comparable} ({comparable/len(informative):.1%})")

Both sides censored: 65 / 925 (7.0%)
  Of those, ratio == exactly 1.0 (same cap both sides, uninformative): 43
At least one side censored: 179 / 925 (19.4%)

Excluding both-censored rows (860 compounds remain):
  More potent against WT (>2-fold):      347 (40.3%)
  More potent against D816V (>2-fold):   235 (27.3%)
  Comparable potency (0.5x-2x):           278 (32.3%)


## 2. Check the two named anchors from Design Doc §4.3

- **Dasatinib**: ~37 nM (D816V) vs. ~79 nM (WT) -- comparable potency both ways (literature values).
- **Imatinib**: known to lose efficacy against D816V relative to WT.

Note up front: these literature values are external reference points (§4.3), not necessarily reproduced by ChEMBL's own aggregated measurements in this dataset -- checking that is exactly what this section does.

In [6]:
imatinib_id = "CHEMBL941"
dasatinib_id = "CHEMBL1421"

if imatinib_id in pairs.index:
    row = pairs.loc[imatinib_id]
    print(f"Imatinib: p_value_wt={row['p_value_wt']:.3f}, p_value_d816v={row['p_value_d816v']:.3f}, "
          f"selectivity_ratio={row['selectivity_ratio']:.2f} (>1 means more potent against WT)")
else:
    print("Imatinib not found in the paired table.")

if dasatinib_id in pairs.index:
    row = pairs.loc[dasatinib_id]
    print(f"Dasatinib: selectivity_ratio={row['selectivity_ratio']:.2f}")
else:
    print("Dasatinib has no D816V row in the cleaned dataset (didn't survive Phase 2's aggregation), "
          "so no ratio can be computed from our own data -- the §4.3 anchor values for it are external "
          "literature references, not something this dataset can independently confirm.")

Imatinib: p_value_wt=6.967, p_value_d816v=6.009, selectivity_ratio=9.07 (>1 means more potent against WT)
Dasatinib has no D816V row in the cleaned dataset (didn't survive Phase 2's aggregation), so no ratio can be computed from our own data -- the §4.3 anchor values for it are external literature references, not something this dataset can independently confirm.


Imatinib's ratio from our own data (computed above) is directionally consistent with the known "loses efficacy against D816V" pattern (ratio > 1 = more potent against WT) -- a first, informal confirmation, ahead of the formal anchor-point check in step 3.

## 3. Save the paired/selectivity table

Reproducible by re-running this notebook (deterministic, no randomness involved), so gitignored like the other `data/processed/` artifacts.

In [7]:
pairs_path = PROCESSED_DIR / "selectivity_pairs.csv"
pairs.to_csv(pairs_path)
print(f"Saved {pairs_path} ({len(pairs)} rows)")

Saved ../data/processed/selectivity_pairs.csv (925 rows)


### Summary

- 925 compounds have both a WT and a D816V bioactivity record in the cleaned dataset; selectivity ratio computed for all of them, saved to `data/processed/selectivity_pairs.csv`.
- Selectivity is broadly distributed rather than concentrated near 1 -- worth keeping in mind that "selectivity" is a real, variable property across this compound set, not a rare exception.
- **Data-quality caveat, checked directly rather than assumed away:** 65/925 (7.0%) have both sides censored, and 43 of those land at exactly ratio=1.0 purely because both hit the same censoring cap -- not real evidence of equipotency. `both_censored` is now a first-class column so downstream analysis can filter these out; excluding them shifts the "comparable potency" bucket from 34.8% to 32.3% (860 compounds remain).
- Imatinib (in the paired table): ratio > 1, directionally consistent with its known loss of efficacy against D816V.
- Dasatinib: **not** in the paired table -- it has no D816V row in the cleaned dataset, so the §4.3 anchor values for it remain an external literature reference, not something independently reproduced from our own ChEMBL pull. Worth stating explicitly rather than silently treating it as covered.
- Next (not yet done in this pass): apply the Phase 4/addendum potency model separately to WT- and mutant-labeled subsets (step 2), then the formal anchor-point direction check (step 3).